In [1]:
import os
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.decomposition import PCA

In [2]:
import os

os.makedirs("Models", exist_ok=True)

print("Models folder is ready.")

Models folder is ready.


In [3]:
############################################################
# CONFIG
############################################################

TRAIN_DIR = "/kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/trainingJSON"
TEST_DIR = "/kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/testingJSON"

TARGET_LEN = 1019
PCA_COMPONENTS = 6

BATCH_SIZE = 32
EPOCHS = 200
LR = 1e-3

############################################################
# LABEL MAP
############################################################

LABEL_MAP = {
    "noGesture": 0,
    "fist": 1,
    "waveIn": 2,
    "waveOut": 3,
    "open": 4,
    "pinch": 5
}

############################################################
# PAD / CROP
############################################################

def pad_or_crop(signal, target_len=TARGET_LEN):

    current_len = signal.shape[1]

    if current_len > target_len:

        signal = signal[:, :target_len]

    elif current_len < target_len:

        pad = target_len - current_len

        signal = np.pad(
            signal,
            ((0, 0), (0, pad)),
            mode='constant'
        )

    return signal

############################################################
# LOAD DATA
############################################################

def load_dataset(folder, allowed_users=None):



    X = []
    y = []

    for root, dirs, files in os.walk(folder):

 

        user_name = os.path.basename(root)

        if allowed_users is not None:
            if user_name not in allowed_users:
               
                continue

        for file in files:

            if not file.endswith(".json"):
                continue

            path = os.path.join(root, file)

        

            with open(path, "r") as f:
                data = json.load(f)

         

            if "trainingSamples" not in data:
               
                continue

            samples = data["trainingSamples"]

           
            for key in samples:

                sample = samples[key]

                gesture = sample["gestureName"]

                if gesture not in LABEL_MAP:
                    
                    continue

                emg = sample["emg"]

                signal = np.array([
                    emg["ch1"],
                    emg["ch2"],
                    emg["ch3"],
                    emg["ch4"],
                    emg["ch5"],
                    emg["ch6"],
                    emg["ch7"],
                    emg["ch8"]
                ], dtype=np.float32)

                signal = pad_or_crop(signal)

                X.append(signal)
                y.append(LABEL_MAP[gesture])


    return np.array(X, dtype=np.float32), np.array(y)

############################################################
# PCA
############################################################

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def apply_pca(X_train, X_test):

    scaler = StandardScaler()

    pca = PCA(
        n_components=PCA_COMPONENTS
    )

    temp = X_train.transpose(0,2,1)
    temp = temp.reshape(-1,8)

    temp = scaler.fit_transform(temp)

    pca.fit(temp)

    train_out = []

    for sample in X_train:

        sample = sample.T
        sample = scaler.transform(sample)
        sample = pca.transform(sample)
        sample = sample.T

        train_out.append(sample)

    test_out = []

    for sample in X_test:

        sample = sample.T
        sample = scaler.transform(sample)
        sample = pca.transform(sample)
        sample = sample.T

        test_out.append(sample)

    return (
        np.array(train_out, dtype=np.float32),
        np.array(test_out, dtype=np.float32)
    )

In [4]:
############################################################
# DATASET
############################################################

class EMGDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

############################################################
# CNN MODEL
############################################################

In [5]:
############################################################
# CNN MODEL
############################################################

class PCA_CNN(nn.Module):

    def __init__(self):

        super().__init__()

        ############################################################
        # Conv1
        ############################################################

        self.conv1 = nn.Conv1d(
            in_channels=6,
            out_channels=32,
            kernel_size=2
        )

        self.pool = nn.MaxPool1d(
            kernel_size=2
        )

        ############################################################
        # Conv2 (64 -> 52 filters)
        ############################################################

        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=2
        )

        ############################################################
        # GAP
        ############################################################

        self.gap = nn.AdaptiveAvgPool1d(1)

        ############################################################
        # FC
        ############################################################

        self.fc1 = nn.Linear(
            32,
            64
        )

        self.fc2 = nn.Linear(
            64,
            6
        )

    def forward(self, x):

        x = self.conv1(x)
        x = F.relu(x)

        x = self.pool(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = self.gap(x)

        x = x.squeeze(-1)

        x = self.fc1(x)
        x = F.relu(x)

        x = self.fc2(x)

        return x

In [6]:
############################################################
# LOAD DATA
############################################################
TEST_USERS = [
    "user1","user2","user3","user4","user5",
    "user6","user7","user8","user9","user10",
    "user11","user12","user13","user14","user15",
    "user16","user17","user18","user19","user20",
    "user21"
]
print("Loading data...")

print("TRAIN_DIR =", TRAIN_DIR)
print("TEST_DIR  =", TEST_DIR)

X_train, y_train = load_dataset(TRAIN_DIR)

X_test, y_test = load_dataset(
    TEST_DIR,
    TEST_USERS
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("Train distribution:")
print(np.unique(y_train, return_counts=True))

print("Test distribution:")
print(np.unique(y_test, return_counts=True))

############################################################
# PCA
############################################################

print("Applying PCA...")

X_train, X_test = apply_pca(
    X_train,
    X_test
)

np.save("X_train_pca.npy", X_train)
np.save("y_train.npy", y_train)

np.save("X_test_pca.npy", X_test)
np.save("y_test.npy", y_test)

print("After PCA")

print("Train :", X_train.shape)
print("Test  :", X_test.shape)

############################################################
# DATALOADER
############################################################

train_dataset = EMGDataset(
    X_train,
    y_train
)

test_dataset = EMGDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

Loading data...
TRAIN_DIR = /kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/trainingJSON
TEST_DIR  = /kaggle/input/datasets/vikas635233/dataset-for-train-test/EMG-EPN612 Dataset/testingJSON
Train shape: (45900, 8, 1019)
Test shape : (3150, 8, 1019)
X_train: (45900, 8, 1019)
y_train: (45900,)
X_test : (3150, 8, 1019)
y_test : (3150,)
Train distribution:
(array([0, 1, 2, 3, 4, 5]), array([7650, 7650, 7650, 7650, 7650, 7650]))
Test distribution:
(array([0, 1, 2, 3, 4, 5]), array([525, 525, 525, 525, 525, 525]))
Applying PCA...
After PCA
Train : (45900, 6, 1019)
Test  : (3150, 6, 1019)


In [7]:
############################################################
# DEVICE
############################################################

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

############################################################
# MODEL
############################################################

model = PCA_CNN().to(device)

total_params = sum(p.numel() for p in model.parameters())
print("Total Parameters:", total_params)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

############################################################
# TRAIN
############################################################

best_acc = 0

for epoch in range(EPOCHS):

    ########################################################
    # TRAIN
    ########################################################

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(
            outputs,
            y
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        preds = outputs.argmax(1)

        train_total += y.size(0)

        train_correct += (
            preds == y
        ).sum().item()

    train_acc = (
        100.0 *
        train_correct /
        train_total
    )

    ########################################################
    # TEST
    ########################################################

    model.eval()

    test_correct = 0
    test_total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = outputs.argmax(1)

            test_total += y.size(0)

            test_correct += (
                preds == y
            ).sum().item()

    test_acc = (
        100.0 *
        test_correct /
        test_total
    )

    ########################################################
    # PRINT RESULTS
    ########################################################

    print(
        f"Epoch {epoch+1:03d} | "
        f"Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | "
        f"Test Acc {test_acc:.2f}%"
    )

    ########################################################
    # SAVE BEST MODEL
    ########################################################

    if test_acc > best_acc:

        best_acc = test_acc

        torch.save(
            model.state_dict(),
            "Models/best_model_conv32.pth"
        )

        print(
            f"Saved model : {best_acc:.2f}%"
        )

print("\nTraining Complete")
print("Best Test Accuracy =", best_acc)

Device: cuda
Total Parameters: 4998
Epoch 001 | Loss 1236.8919 | Train Acc 67.98% | Test Acc 84.95%
Saved model : 84.95%
Epoch 002 | Loss 687.2482 | Train Acc 83.20% | Test Acc 88.41%
Saved model : 88.41%
Epoch 003 | Loss 581.4027 | Train Acc 85.45% | Test Acc 89.62%
Saved model : 89.62%
Epoch 004 | Loss 528.5416 | Train Acc 86.62% | Test Acc 89.11%
Epoch 005 | Loss 490.1065 | Train Acc 87.62% | Test Acc 90.10%
Saved model : 90.10%
Epoch 006 | Loss 460.5866 | Train Acc 88.29% | Test Acc 91.40%
Saved model : 91.40%
Epoch 007 | Loss 439.8658 | Train Acc 88.83% | Test Acc 91.40%
Epoch 008 | Loss 419.7712 | Train Acc 89.31% | Test Acc 91.59%
Saved model : 91.59%
Epoch 009 | Loss 405.2189 | Train Acc 89.79% | Test Acc 91.24%
Epoch 010 | Loss 392.8070 | Train Acc 90.04% | Test Acc 91.94%
Saved model : 91.94%
Epoch 011 | Loss 382.1208 | Train Acc 90.30% | Test Acc 92.95%
Saved model : 92.95%
Epoch 012 | Loss 371.0605 | Train Acc 90.59% | Test Acc 91.90%
Epoch 013 | Loss 360.6101 | Train Acc 9

In [8]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

model.load_state_dict(
    torch.load("Models/best_model_conv32.pth")
)

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = model(x)

        preds = outputs.argmax(1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("Confusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

Confusion Matrix
[[524   1   0   0   0   0]
 [  0 504   1   0   5  15]
 [  0   2 518   1   1   3]
 [  2   0   5 482  34   2]
 [  0   7   0  31 470  17]
 [  1  11   1   1  27 484]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       525
           1       0.96      0.96      0.96       525
           2       0.99      0.99      0.99       525
           3       0.94      0.92      0.93       525
           4       0.88      0.90      0.89       525
           5       0.93      0.92      0.93       525

    accuracy                           0.95      3150
   macro avg       0.95      0.95      0.95      3150
weighted avg       0.95      0.95      0.95      3150



In [9]:
############################################################
# WEIGHT QUANTIZATION
############################################################
def quantize_weights(weights, bits):

    qmin = -(2 ** (bits - 1))
    qmax = (2 ** (bits - 1)) - 1

    # Linear layer
    if weights.ndim == 2:

        out = torch.empty_like(weights)

        for i in range(weights.size(0)):

            w = weights[i]

            max_val = w.abs().max()

            if max_val == 0:
                out[i] = w
                continue

            scale = max_val / qmax

            q = torch.round(w / scale)
            q = torch.clamp(q, qmin, qmax)

            out[i] = q * scale

        return out

    # Conv1D layer
    elif weights.ndim == 3:

        out = torch.empty_like(weights)

        for i in range(weights.size(0)):

            w = weights[i]

            max_val = w.abs().max()

            if max_val == 0:
                out[i] = w
                continue

            scale = max_val / qmax

            q = torch.round(w / scale)
            q = torch.clamp(q, qmin, qmax)

            out[i] = q * scale

        return out

    # Bias or scalar
    else:

        max_val = weights.abs().max()

        if max_val == 0:
            return weights.clone()

        scale = max_val / qmax

        q = torch.round(weights / scale)
        q = torch.clamp(q, qmin, qmax)

        return q * scale

In [10]:
############################################################
# UNIFORM 8-BIT QUANTIZATION
############################################################

import copy
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

############################################################
# LOAD BEST MODEL
############################################################

base_model = PCA_CNN().to(device)

base_model.load_state_dict(
    torch.load("Models/best_model_conv32.pth")
)

base_model.eval()

############################################################
# CREATE UNIFORM 8-BIT MODEL
############################################################

uniform8_model = copy.deepcopy(base_model)

with torch.no_grad():

    for name, param in uniform8_model.named_parameters():

        if "weight" in name:

            param.data = quantize_weights(
                param.data,
                bits=8
            )

############################################################
# EVALUATE
############################################################

uniform8_model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = uniform8_model(x)

        preds = outputs.argmax(1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

acc = 100 * np.mean(
    np.array(y_true) ==
    np.array(y_pred)
)

print("="*60)
print("UNIFORM 8-BIT QUANTIZATION")
print("="*60)
print(f"Accuracy = {acc:.4f}%")

print("\nConfusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

UNIFORM 8-BIT QUANTIZATION
Accuracy = 94.6032%

Confusion Matrix
[[524   1   0   0   0   0]
 [  0 504   1   0   5  15]
 [  0   2 518   1   1   3]
 [  2   0   5 481  35   2]
 [  0   7   1  31 469  17]
 [  1  10   1   1  28 484]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       525
           1       0.96      0.96      0.96       525
           2       0.98      0.99      0.99       525
           3       0.94      0.92      0.93       525
           4       0.87      0.89      0.88       525
           5       0.93      0.92      0.93       525

    accuracy                           0.95      3150
   macro avg       0.95      0.95      0.95      3150
weighted avg       0.95      0.95      0.95      3150



In [11]:
############################################################
# UNIFORM 6-BIT QUANTIZATION
############################################################

import copy
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

base_model = PCA_CNN().to(device)

base_model.load_state_dict(
    torch.load("Models/best_model_conv32.pth")
)

base_model.eval()

uniform6_model = copy.deepcopy(base_model)

with torch.no_grad():

    for name, param in uniform6_model.named_parameters():

        if "weight" in name:

            param.data = quantize_weights(
                param.data,
                bits=6
            )

uniform6_model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)

        outputs = uniform6_model(x)

        preds = outputs.argmax(1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

acc = 100*np.mean(
    np.array(y_true)==
    np.array(y_pred)
)

print("="*60)
print("UNIFORM 6-BIT QUANTIZATION")
print("="*60)
print(f"Accuracy = {acc:.4f}%")

print("\nConfusion Matrix")
print(confusion_matrix(y_true,y_pred))

print("\nClassification Report")
print(classification_report(y_true,y_pred))

UNIFORM 6-BIT QUANTIZATION
Accuracy = 94.9524%

Confusion Matrix
[[524   1   0   0   0   0]
 [  0 492   1   0   6  26]
 [  0   1 519   1   1   3]
 [  3   0   4 479  37   2]
 [  0   4   1  22 475  23]
 [  1   6   1   2  13 502]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       525
           1       0.98      0.94      0.96       525
           2       0.99      0.99      0.99       525
           3       0.95      0.91      0.93       525
           4       0.89      0.90      0.90       525
           5       0.90      0.96      0.93       525

    accuracy                           0.95      3150
   macro avg       0.95      0.95      0.95      3150
weighted avg       0.95      0.95      0.95      3150



In [12]:
############################################################
# LAYER-WISE QUANTIZATION SENSITIVITY
############################################################

import copy



def evaluate_model(model):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            preds = outputs.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return 100.0 * correct / total


############################################################
# LOAD BEST MODEL
############################################################

base_model = PCA_CNN().to(device)

base_model.load_state_dict(
    torch.load("Models/best_model_conv32.pth")
)

baseline_acc = evaluate_model(base_model)

print(f"\nBaseline Accuracy = {baseline_acc:.4f}%")


Baseline Accuracy = 94.6667%


In [13]:
############################################################
# TEST EACH LAYER
############################################################

layers_to_test = [
    "conv1.weight",
    "conv2.weight",
    "fc1.weight",
    "fc2.weight"
]

bitwidths = [8, 6, 4, 2]

results = {}

for layer_name in layers_to_test:

    results[layer_name] = {}

    print("\n" + "="*60)
    print("Testing:", layer_name)

    for bits in bitwidths:

        model = copy.deepcopy(base_model)

        with torch.no_grad():

            for name, param in model.named_parameters():

                if name == layer_name:

                    param.data = quantize_weights(
                        param.data,
                        bits
                    )

        acc = evaluate_model(model)

        results[layer_name][bits] = acc

        print(
            f"{bits:2d}-bit --> {acc:.4f}%"
        )


Testing: conv1.weight
 8-bit --> 94.6032%
 6-bit --> 94.6349%
 4-bit --> 94.5397%
 2-bit --> 84.4127%

Testing: conv2.weight
 8-bit --> 94.6349%
 6-bit --> 94.6349%
 4-bit --> 91.5556%
 2-bit --> 42.0635%

Testing: fc1.weight
 8-bit --> 94.5714%
 6-bit --> 94.5714%
 4-bit --> 92.6349%
 2-bit --> 33.7460%

Testing: fc2.weight
 8-bit --> 94.6667%
 6-bit --> 94.1905%
 4-bit --> 90.0635%
 2-bit --> 49.8730%


In [14]:
############################################################
# PRINT TABLE
############################################################

print("\n")
print("="*70)
print("HAQ SENSITIVITY TABLE")
print("="*70)

for layer, vals in results.items():

    print(
        f"{layer:15s}",
        end=""
    )

    for bits in [8,6,4,2]:

        print(
            f"{vals[bits]:8.2f}",
            end=""
        )

    print()

print("\nColumns = [8bit 6bit 4bit 2bit]")



HAQ SENSITIVITY TABLE
conv1.weight      94.60   94.63   94.54   84.41
conv2.weight      94.63   94.63   91.56   42.06
fc1.weight        94.57   94.57   92.63   33.75
fc2.weight        94.67   94.19   90.06   49.87

Columns = [8bit 6bit 4bit 2bit]


In [15]:
############################################################
# EXHAUSTIVE MIXED-PRECISION SEARCH
############################################################

import copy
import itertools
import pandas as pd

layers = [
    "conv1.weight",
    "conv2.weight",
    "fc1.weight",
    "fc2.weight"
]

bit_choices = [8, 6, 4]

results = []

total = len(bit_choices) ** len(layers)

count = 1

for bits in itertools.product(bit_choices, repeat=4):

    print(f"Testing {count}/{total} : {bits}")

    model = copy.deepcopy(base_model)

    with torch.no_grad():

        for name, param in model.named_parameters():

            if name in layers:

                idx = layers.index(name)

                param.data = quantize_weights(
                    param.data,
                    bits[idx]
                )

    ########################################################
    # EVALUATE
    ########################################################

    acc = evaluate_model(model)

    ########################################################
    # DEBUG FOR (8,8,8,8)
    ########################################################

    if bits == (8, 8, 8, 8):

        print("\nUniform from exhaustive search =", acc)

        print("\nWeight sums:")

        for name, param in model.named_parameters():

            if "weight" in name:

                print(name, torch.sum(param).item())

    ########################################################
    # SAVE RESULT
    ########################################################

    results.append({
        "Conv1": bits[0],
        "Conv2": bits[1],
        "FC1": bits[2],
        "FC2": bits[3],
        "Accuracy": acc
    })

    count += 1

############################################################
# CREATE DATAFRAME
############################################################

results_df = pd.DataFrame(results)

print("\nNumber of results:", len(results))

print("\n(8,8,8,8) row:")

print(
    results_df[
        (results_df["Conv1"] == 8) &
        (results_df["Conv2"] == 8) &
        (results_df["FC1"] == 8) &
        (results_df["FC2"] == 8)
    ]
)

Testing 1/81 : (8, 8, 8, 8)

Uniform from exhaustive search = 94.60317460317461

Weight sums:
conv1.weight -11.523223876953125
conv2.weight -107.64735412597656
fc1.weight -79.43236541748047
fc2.weight -83.99010467529297
Testing 2/81 : (8, 8, 8, 6)
Testing 3/81 : (8, 8, 8, 4)
Testing 4/81 : (8, 8, 6, 8)
Testing 5/81 : (8, 8, 6, 6)
Testing 6/81 : (8, 8, 6, 4)
Testing 7/81 : (8, 8, 4, 8)
Testing 8/81 : (8, 8, 4, 6)
Testing 9/81 : (8, 8, 4, 4)
Testing 10/81 : (8, 6, 8, 8)
Testing 11/81 : (8, 6, 8, 6)
Testing 12/81 : (8, 6, 8, 4)
Testing 13/81 : (8, 6, 6, 8)
Testing 14/81 : (8, 6, 6, 6)
Testing 15/81 : (8, 6, 6, 4)
Testing 16/81 : (8, 6, 4, 8)
Testing 17/81 : (8, 6, 4, 6)
Testing 18/81 : (8, 6, 4, 4)
Testing 19/81 : (8, 4, 8, 8)
Testing 20/81 : (8, 4, 8, 6)
Testing 21/81 : (8, 4, 8, 4)
Testing 22/81 : (8, 4, 6, 8)
Testing 23/81 : (8, 4, 6, 6)
Testing 24/81 : (8, 4, 6, 4)
Testing 25/81 : (8, 4, 4, 8)
Testing 26/81 : (8, 4, 4, 6)
Testing 27/81 : (8, 4, 4, 4)
Testing 28/81 : (6, 8, 8, 8)
Testi

In [16]:
for name, param in base_model.named_parameters():
    print(name)

conv1.weight
conv1.bias
conv2.weight
conv2.bias
fc1.weight
fc1.bias
fc2.weight
fc2.bias


In [17]:
############################################################
# CREATE TABLE
############################################################

results_df = pd.DataFrame(
    results,
    columns=[
        "Conv1",
        "Conv2",
        "FC1",
        "FC2",
        "Accuracy"
    ]
)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

results_df = results_df.reset_index(drop=True)

print(results_df)

    Conv1  Conv2  FC1  FC2   Accuracy
0       8      6    6    6  95.015873
1       6      6    6    6  94.952381
2       6      8    6    6  94.857143
3       8      8    6    6  94.825397
4       6      8    8    8  94.730159
..    ...    ...  ...  ...        ...
76      4      8    4    4  86.825397
77      6      8    4    4  86.349206
78      8      8    4    4  86.253968
79      6      6    4    4  86.190476
80      8      6    4    4  86.158730

[81 rows x 5 columns]


In [18]:
results_df.to_csv(
    "Mixed_Quantization_Results.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [19]:
############################################################
# HARDWARE COST ANALYSIS
############################################################

import numpy as np

PARAMS = {
    "Conv1":416,
    "Conv2":2080,
    "FC1":2112,
    "FC2":390
}

# FP32 reference
FP32_BITS = (
    (PARAMS["Conv1"] +
     PARAMS["Conv2"] +
     PARAMS["FC1"] +
     PARAMS["FC2"]) * 32
)

# Calculate hardware cost
results_df["Memory(bits)"] = (
    results_df["Conv1"] * PARAMS["Conv1"] +
    results_df["Conv2"] * PARAMS["Conv2"] +
    results_df["FC1"]   * PARAMS["FC1"] +
    results_df["FC2"]   * PARAMS["FC2"]
)

results_df["Memory(KB)"] = (
    results_df["Memory(bits)"] / 8 / 1024
)

results_df["Compression"] = (
    FP32_BITS / results_df["Memory(bits)"]
)

# Xilinx BRAM18 = 18 Kbits = 18432 bits
results_df["BRAM18"] = np.ceil(
    results_df["Memory(bits)"] / 18432
).astype(int)

############################################################
# KEEP ONLY GOOD MODELS
############################################################

good_models = results_df[
    results_df["Accuracy"] >= 93.0
].copy()

good_models = good_models.sort_values(
    by=["Accuracy", "Memory(bits)"],
    ascending=[False, True]
)

good_models = good_models.reset_index(drop=True)

print(good_models)

    Conv1  Conv2  FC1  FC2   Accuracy  Memory(bits)  Memory(KB)  Compression  \
0       8      6    6    6  95.015873         30820    3.762207     5.189358   
1       6      6    6    6  94.952381         29988    3.660645     5.333333   
2       6      8    6    6  94.857143         34148    4.168457     4.683613   
3       8      8    6    6  94.825397         34980    4.270020     4.572213   
4       6      8    8    8  94.730159         39152    4.779297     4.085002   
5       4      6    6    6  94.698413         29156    3.559082     5.485526   
6       8      6    6    8  94.698413         31600    3.857422     5.061266   
7       6      8    6    8  94.666667         34928    4.263672     4.579020   
8       6      6    6    8  94.634921         30768    3.755859     5.198128   
9       8      6    8    6  94.603175         35044    4.277832     4.563863   
10      8      8    8    8  94.603175         39984    4.880859     4.000000   
11      4      6    6    8  94.571429   

In [20]:
############################################################
# SAVE RESULTS
############################################################

good_models.to_csv(
    "Hardware_Aware_Results.csv",
    index=False
)

print("Saved Successfully!")

print("\nTop Hardware-Aware Models\n")
print(good_models)

Saved Successfully!

Top Hardware-Aware Models

    Conv1  Conv2  FC1  FC2   Accuracy  Memory(bits)  Memory(KB)  Compression  \
0       8      6    6    6  95.015873         30820    3.762207     5.189358   
1       6      6    6    6  94.952381         29988    3.660645     5.333333   
2       6      8    6    6  94.857143         34148    4.168457     4.683613   
3       8      8    6    6  94.825397         34980    4.270020     4.572213   
4       6      8    8    8  94.730159         39152    4.779297     4.085002   
5       4      6    6    6  94.698413         29156    3.559082     5.485526   
6       8      6    6    8  94.698413         31600    3.857422     5.061266   
7       6      8    6    8  94.666667         34928    4.263672     4.579020   
8       6      6    6    8  94.634921         30768    3.755859     5.198128   
9       8      6    8    6  94.603175         35044    4.277832     4.563863   
10      8      8    8    8  94.603175         39984    4.880859     4.00

In [21]:
print("Conv1 :", model.conv1.weight.numel())
print("Conv2 :", model.conv2.weight.numel())
print("FC1   :", model.fc1.weight.numel())
print("FC2   :", model.fc2.weight.numel())

Conv1 : 384
Conv2 : 2048
FC1   : 2048
FC2   : 384
